# Setup

# This file requires using the environment created from environment-gtfparse.yml

In [ ]:
# Imports

import polars as pl
import duckdb
from pathlib import Path
import math
import os
import numpy as np
from tqdm.notebook import tqdm 

For this analysis you will need to obtain the COSMIC genome screening files as well as the COSMIC mutation classification file.
This analysis was based on the v99 release of COSMIC database
You can find the files and the license information at
https://cancer.sanger.ac.uk/cosmic/download/cosmic


Create a directory `COSMIC` inside the `data/subsidiary-files` folder and copy the Cosmic tsv files there.

In [ ]:
# Paths
project_dir = Path("/data/teamgdansk/mwaleron/carmen-analysis")

data_dir = project_dir.joinpath("data")
temp_dir = project_dir.joinpath("temp")

db_main_file = data_dir.joinpath("carmen-main.parquet")

data_subs_dir = data_dir.joinpath("subsidiary-files")
data_pub_dir = data_dir.joinpath("to-be-published")

cosmic_data_dir = data_subs_dir.joinpath("COSMIC")
cosmic_genome_screening_file = cosmic_data_dir.joinpath("Cosmic_GenomeScreensMutant_v99_GRCh38.tsv")
cosmic_classification_file = cosmic_data_dir.joinpath("Cosmic_Classification_v99_GRCh38.tsv")

braun_dataset_mut = data_subs_dir.joinpath("Braun_mutations_hg38_with_epitope_contigs.tsv")


gendode_data_dir = data_subs_dir.joinpath("GENCODE")
gencodev39_stripped = gendode_data_dir.joinpath("gencode.v39.annotation.stripped.versions.parquet")

contig_scaffold_list_file = data_pub_dir.joinpath("scaff_all_expanded.tsv")
contig_peptide_list_file = data_pub_dir.joinpath("contig_unique_peptide_list.tsv")
contig_hotspot_score_file = data_subs_dir.joinpath("sorted_normalized_contig_score.parquet")
contig_hotspot_score_top_quadrant_file = data_subs_dir.joinpath("contigs_top_quadrant_by_peptides_coverage.parquet")
pogo_mapping_filtered_by_top_quadrant_contigs = data_subs_dir.joinpath("pogo_tq.parquet")

exon_ranges_in_contigs_file = data_subs_dir.joinpath("evenbiggerjoin_cause_its_exons_not_just_genes_v2_just_the_contig_genes.parquet")

cosmic_somatics_overlap_with_contigs_file = data_subs_dir.joinpath("cosmic_somatics_overlap_with_contigs.parquet")
cosmic_somatics_overlap_with_top_quadrant_file = data_subs_dir.joinpath("cosmic_somatics_overlap_with_tq.parquet")
cosmic_mutations_in_contigs_semi_anti_file_B = data_subs_dir.joinpath("contig_cosmic_mutations.parquet")
cosmic_mutations_in_contigs_semi_anti_file_B_compliment = data_subs_dir.joinpath("non_contig_cosmic_mutations.parquet")
cosmic_mutations_in_top_quadrant_semi_anti_file_B = data_subs_dir.joinpath("tq_cosmic_mutations.parquet")
cosmic_mutations_in_top_quadrant_semi_anti_file_B_compliment = data_subs_dir.joinpath("non_tq_cosmic_mutations.parquet")

main_samples_file = data_dir.joinpath("main-output-table-1.tsv")
main_samples_file_2 = data_dir.joinpath("main-output-table-2.tsv")
main_samples_file_3 = data_subs_dir.joinpath("emilia_umap_with_ids.parquet")

In [6]:
cosmic_somatics = pl.read_csv(
    cosmic_genome_screening_file, 
    separator='\t')
cosmic_classification = pl.read_csv(
    cosmic_classification_file, 
    separator='\t')
contig_scaffold_list = pl.read_csv(
    contig_scaffold_list_file,
    separator='\t')
contig_peptide_list = pl.read_csv(
    contig_peptide_list_file, 
    separator='\t')

In [4]:
#exec disabled - just read the saved parquet
%%script false --no-raise-error

justcontigs = contig_scaffold_list.filter(pl.col("gap_width") == 0)
mutations_in_contigs = duckdb.sql('''
                                select *
                                from justcontigs j, cosmic_somatics_chr c
                                where (j.Chromosome = c.CHROMOSOME)
                                and(
                                    (c.GENOME_START between j.Gene_start and j.Gene_end)
                                or  (c.GENOME_STOP between j.Gene_Start and j.Gene_end)
                                )
                                ''')
mutations_in_contigs_res = mutations_in_contigs.pl()
mutations_in_contigs_res.write_parquet(cosmic_somatics_overlap_with_contigs_file)

UsageError: Line magic function `%%script` not found.


In [8]:
cosmic_mutcount_per_contig_tmp = temp_dir.joinpath("mutation_count_per_contig.parquet")

mutations_in_contigs_res = pl.read_parquet(cosmic_somatics_overlap_with_contigs_file)
mutation_count_per_contig = mutations_in_contigs_res\
                            .group_by("Id").len().sort("len")
mutation_count_per_contig_C_name =  contig_scaffold_list\
                                    .join(
                                       mutation_count_per_contig,
                                       on="Id",
                                       how="left")\
                                    .fill_null(0)\
                                    .filter(pl.col("gap_width") == 0)\
                                    .select(["Contigs", "len"])
mutation_sum_per_scaffold = contig_scaffold_list\
                            .join(
                                mutation_count_per_contig_C_name,
                                on="Contigs")\
                            .select(["Id", "len"])\
                            .group_by("Id").sum().sort("len")
mutation_sum_per_scaffold.filter(pl.col("Id").str.starts_with("S_0"))\
                         .write_parquet(cosmic_mutcount_per_contig_tmp)

/tmp/ipykernel_104810/2108381767.py:6: DeprecationWarning: The default coalesce behavior of left join will change to `False` in the next breaking release. Pass `coalesce=True` to keep the current behavior and silence this warning.
  mutation_count_per_contig_C_name =  contig_scaffold_list\


In [5]:
unique_peptides_per_contig_tmp = temp_dir.joinpath("unique_peptides_per_contig.parquet")
contig_peptide_one_to_one = contig_peptide_list\
                            .with_columns(pl.col("pep_list").str.split(','))\
                            .explode("pep_list")\
                            .unique(["pep_list", "contig"])\
                            .select(["pep_list", "contig"])
scaffold_peptide = contig_scaffold_list\
                   .join(
                       contig_peptide_one_to_one,
                       left_on="Contigs",
                       right_on="contig")\
                   .unique(["Id", "pep_list"])\
                   .group_by("Id").agg("pep_list")
emilia_umap = pl.read_csv(main_samples_file, separator="\t")
emilia_umap_peptides = pl.read_csv(main_samples_file_2, separator="\t")
emilia_umap_with_ids =  emilia_umap\
                        .with_row_index()\
                        .hstack(emilia_umap_peptides.select("Id", "Peptides"))\
                        .with_columns(Peptides = pl.col("Peptides").str.split(","))
emilia_umap_with_ids.write_parquet(main_samples_file_3)
unique_peptides_per_contig =    scaffold_peptide\
                                .with_columns(unique_peptides = pl.col("pep_list").list.len())\
                                .filter(pl.col("Id").str.starts_with("S_0"))
unique_peptides_per_contig.write_parquet(unique_peptides_per_contig_tmp)

calculation_spam_dir = temp_dir.joinpath("parquetspam")
calculation_spam_dir.mkdir(parents=True,exist_ok=True)
contig_peptide = scaffold_peptide.filter(pl.col("Id").str.starts_with("S_0"))
number_of_shards = math.ceil(len(contig_peptide)/10000)
for i in range(number_of_shards):
    contig_peptide.slice(10000*i, 10000)\
                  .write_parquet(calculation_spam_dir.joinpath(f"contig_slice_{i}.parquet"))

In [ ]:
# run the compute on the slices
# it is recommended to use parallel, SLURM or other parallelization method here,
# eg seq 0 1 {number_of_shards} | parallel --bar python ./calculate_immune_score.py
for i in range(number_of_shards):
    os.system(f"python ./calculate_immune_score.py {i}")

In [ ]:
contig_hotspot_score_file_tmp = temp_dir.joinpath("hotspot_score_3_axis_per_contig.parquet")
test=pl.read_parquet(calculation_spam_dir.joinpath("scored_contig_slice_*"))
popcov_per_contig_file_tmp = temp_dir.joinpath("population_coverage_per_contig.parquet")
test.write_parquet(popcov_per_contig_file_tmp)
mutation_sum_per_scaffold = pl.read_parquet(cosmic_mutcount_per_contig_tmp)
test.join(unique_peptides_per_contig, on="Id")\
    .join(mutation_sum_per_scaffold, on="Id")\
    .drop("pep_list", "pep_list_right")\
    .rename({"len":"mutation_count"})\
    .write_parquet(contig_hotspot_score_file_tmp)
exon_join=pl.scan_parquet(exon_ranges_in_contigs_file)
contig_exon_len = exon_join\
                    .filter(pl.col("gap_width")==0)\
                    .with_columns(exon_length = pl.col("exon_end") - pl.col("exon_start"))\
                    .select("Id", "exon_length")\
                    .group_by("Id").sum().collect()
hotspot_score_3_axis_per_contig = pl.read_parquet(contig_hotspot_score_file_tmp).join(contig_exon_len, on="Id")
hotspot_score_3_axis_per_contig =  hotspot_score_3_axis_per_contig\
                                   .with_columns(normalized_unique_pep = pl.col("unique_peptides") / pl.col("exon_length"))
hotspot_score_3_axis_per_contig = hotspot_score_3_axis_per_contig\
                                  .sort(pl.col("popcov_but_sqrt"), descending=True)\
                                  .with_row_index("pop_cov_idx")\
                                  .sort(pl.col("mutation_count"), descending=True)\
                                  .with_row_index("mut_count_idx")\
                                  .sort(pl.col("normalized_unique_pep"), descending=True)\
                                  .with_row_index("unique_pep_idx")\
                                  .with_columns(
                                      ranking= pl.col("unique_pep_idx") + pl.col("mut_count_idx") + pl.col("pop_cov_idx"))\
                                  .sort("ranking")
hotspot_score_3_axis_per_contig.write_parquet(contig_hotspot_score_file)
hotspot_score_3_axis_per_contig = pl.read_parquet(contig_hotspot_score_file)
hotspot_score_3_axis_per_contig.filter(
                                        pl.col("unique_pep_idx") < pl.col("unique_pep_idx").median() , 
                                        pl.col("popcov_but_sqrt") > 13)\
                                .write_parquet(contig_hotspot_score_top_quadrant_file)
top_quadrant = pl.read_parquet(contig_hotspot_score_top_quadrant_file)
justcontigs = contig_scaffold_list.filter(pl.col("gap_width") == 0, pl.col("Id").is_in(top_quadrant["Id"]))


In [9]:
pep_main=pl.read_parquet(db_main_file)

In [10]:
pep_main.head()

Study_id,Sample_name,Peptide,Peptide_type,Spectral_count,Assigned_modifications,Haplotype,Netmhcpan_binder,Promiscuity
str,str,str,str,i64,str,str,str,f64
"""MSV000080527-84172-84442""","""sample_A0101""","""AADIFYSRY""","""canonical""",12,null,"""A*01:01""","""Y""",0.688129
"""MSV000080527-84172-84442""","""sample_A0101""","""AADLNLVLY""","""canonical""",14,null,"""A*01:01""","""Y""",8.672632
"""MSV000080527-84172-84442""","""sample_A0101""","""AADPNAAWAAY""","""canonical""",1,null,"""A*01:01""","""Y""",0.387766
"""MSV000080527-84172-84442""","""sample_A0101""","""AADPNAAWAAYY""","""canonical""",7,null,"""A*01:01""","""Y""",0.769507
"""MSV000080527-84172-84442""","""sample_A0101""","""AAEDIINYTEPKLGY""","""canonical""",1,null,"""A*01:01""","""Y""",0.0


In [4]:
emilia_umap_with_ids = pl.read_parquet(main_samples_file_3)
xallcen = (emilia_umap_with_ids["x"].min() + emilia_umap_with_ids["x"].max() )/2
yallcen = (emilia_umap_with_ids["y"].min() + emilia_umap_with_ids["y"].max() )/2

In [ ]:
def promiscuity_peptide_list(input):
    scorebase = emilia_umap_with_ids\
        .with_columns(
        scafres = pl.col("Peptides").list.set_intersection(input
        ))\
        .with_columns(alen = pl.col("scafres").list.len()).sort("alen")
    xy = scorebase.filter(pl.col("alen")>0).select("x","y")
    count = len(xy)
    if count == 0 :
        return 0
    xcen = xy["x"].sum()/count
    ycen = xy["y"].sum()/count
    lth = np.sqrt( np.power(xallcen-xcen,2) + np.power(yallcen-ycen,2))
    popcov_but_sqrt = count / math.sqrt(1+lth)
    return popcov_but_sqrt

In [8]:
czytoprom = []
for umap in tqdm(emilia_umap_with_ids.iter_rows(named=True)):
    czytoprom.append(promiscuity_peptide_list(umap["Peptides"]))
czytoprom

0it [00:00, ?it/s]

[883.910793639971,
 638.8008939022592,
 1208.9796874488784,
 945.3704822881189,
 591.9475192737682,
 895.8843034488854,
 550.8728102494181,
 772.4626799301442,
 670.5295974477691,
 617.9558094411568,
 628.1668716524906,
 632.1804377561988,
 662.5624153811046,
 1974.0232538263913,
 1703.895620620031,
 1641.8219069567078,
 468.3290065753511,
 1138.5123256200625,
 2061.9880471246347,
 1931.411066213954,
 7.63475327056367,
 95.75851286565667,
 93.27702599073969,
 71.35793895151963,
 609.0177556122259,
 938.5179595479666,
 831.9067609103312,
 1158.0623532695054,
 893.1525551940465,
 1072.8162584286072,
 299.4185396581656,
 528.9058693498288,
 448.28054261871443,
 453.3600989088737,
 166.13807502060672,
 297.08990550904423,
 178.8552876909029,
 23.91864794222972,
 261.7239203964983,
 0.5027194866830007,
 244.19528034279375,
 11.71903338828537,
 2585.478069653016,
 1026.7197779525225,
 952.9954111857886,
 989.1127152239825,
 333.97054464098505,
 1074.2112539400005,
 187.49308477489396,
 1060.

In [9]:
sample_prom = emilia_umap_with_ids.hstack(pl.DataFrame({"Promiscuity":czytoprom}))

In [10]:
sample_prom

index,Sample,Binding_alleles,x,y,label,Id,Peptides,Promiscuity
u32,str,str,f64,f64,i64,str,list[str],f64
0,"""sample_PXD027182_NB8_PASEGA_0B…","""A*02:01,A*24:02,B*51:01:01,B*5…",0.191533,2.5076637,3,"""GC_0""","[""AAMPRPVSY"", ""AAYGRSPMV"", … ""YAAPHPLQSY""]",883.910794
1,"""sample_PXD027182_NB8_PASEGA_0B…","""A*02:01,A*24:02,B*51:01:01,B*5…",-1.854416,9.617391,13,"""GC_1""","[""AYASQFGTF"", ""AYGSLFNTI"", … ""YYTPITPHL""]",638.800894
2,"""sample_PXD027182_NB8_PASEGA_0B…","""A*02:01,A*24:02,B*51:01:01,B*5…",5.087642,6.512341,6,"""GC_2""","[""ALPPVLTTV"", ""DPPPGSHVI"", … ""YPVEQMTTI""]",1208.979687
3,"""sample_PXD027182_NB8_PASEGA_0B…","""A*02:01,A*24:02,B*51:01:01,B*5…",-1.499036,1.5007926,10,"""GC_3""","[""AIVDKVPSV"", ""ALADGVQKV"", … ""YQVGQLYSV""]",945.370482
4,"""sample_PXD027182_NB8_PASEGA_0B…","""A*02:01,A*24:02,B*51:01:01,B*5…",4.7801905,5.940344,6,"""GC_4""","[""DAAAKALRI"", ""DAAEFAISI"", … ""YAFPKAVSV""]",591.947519
…,…,…,…,…,…,…,…,…
5688,"""sample_20200513_COL015_00007a_…",null,5.4869995,1.7892371,0,"""GC_5688""","[""AEEYEFLTPVEEAPK"", ""ALQHMTDFAIQFNK"", … ""YGPSSVSFADDFVR""]",56.588294
5689,"""sample_20200513_COL015_00007a_…",null,5.382379,2.0899956,0,"""GC_5689""","[""AAAFEEQENETVVVK"", ""AALSASEGEEVPQDK"", … ""YVAEIEKEKEENEKK""]",107.256242
5690,"""sample_20200513_COL015_00007a_…",null,5.620569,3.7104275,0,"""GC_5690""","[""AGVNVEPFWPGLFAK"", ""AYSNWPTYPQLYVK"", … ""YNEQHVPGSPFTAR""]",32.488022


# Top quadrant set generation

In [25]:
tq_peptides = top_quadrant.join(contig_scaffold_list, on="Id").join(contig_peptide_list, left_on="Contigs", right_on="contig").with_columns(pl.col("pep_list").str.split(",")).explode("pep_list").select(pl.col("pep_list").unique())

In [16]:

pogo_annotations_file = data_dir.joinpath("carmen-mapped-protein-annotations-pogo.parquet")

In [27]:
pl.read_parquet(pogo_annotations_file).filter(pl.col("Peptide").is_in(tq_peptides["pep_list"])).write_parquet(pogo_mapping_filtered_by_top_quadrant_contigs)

In [15]:
pl.read_parquet(pogo_mapping_filtered_by_top_quadrant_contigs)

Peptide,Protein_id,Transcript_id,Gene_id,Protein_start,Protein_end,Chromosome,Gene_start,Gene_end,Strand
str,str,str,str,i64,i64,str,i64,i64,str
"""STFEDPQRLY""","""ENSP00000412228""","""ENST00000455979""","""ENSG00000187634""",29,38,"""1""",939360,939390,"""+"""
"""STFEDPQRLY""","""ENSP00000349216""","""ENST00000341065""","""ENSG00000187634""",126,135,"""1""",939360,939390,"""+"""
"""STFEDPQRLY""","""ENSP00000480870""","""ENST00000618181""","""ENSG00000187634""",186,195,"""1""",939360,939390,"""+"""
"""STFEDPQRLY""","""ENSP00000342313""","""ENST00000342066""","""ENSG00000187634""",202,211,"""1""",939360,939390,"""+"""
"""STFEDPQRLY""","""ENSP00000482090""","""ENST00000617307""","""ENSG00000187634""",203,212,"""1""",939360,939390,"""+"""
…,…,…,…,…,…,…,…,…,…
"""IRQAGGIGK""",null,null,null,0,0,"""Y""",57210701,57210728,"""+"""
"""RQAGGIGK""",null,null,null,0,0,"""Y""",57210704,57210728,"""+"""
"""HLMSDLFNK""",null,null,null,0,0,"""Y""",57211574,57211601,"""+"""


In [ ]:
#exec disabled - just read the saved parquet
%%script false --no-raise-error

cosmic_somatics_chr = pl.read_csv(
    cosmic_genome_screening_file, 
    separator='\t').with_columns(CHROMOSOME = pl.lit("chr")+pl.col("CHROMOSOME").str.replace("MT","M"))
justtq = contig_scaffold_list.filter(pl.col("gap_width") == 0, pl.col("Id").is_in(top_quadrant["Id"]))
mutations_in_tq = duckdb.sql('''
                                select *
                                from justtq j, cosmic_somatics_chr c
                                where (j.Chromosome = c.CHROMOSOME)
                                and(
                                    (c.GENOME_START between j.Gene_start and j.Gene_end)
                                or  (c.GENOME_STOP between j.Gene_Start and j.Gene_end)
                                )
                                ''')
mutations_in_tq_res = mutations_in_tq.pl()
mutations_in_tq_res.write_parquet(cosmic_somatics_overlap_with_top_quadrant_file)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# This is the correct set in the sense mutations aren't doubled 
# Thus this one will be used for statistics
%%script false --no-raise-error
setB = cosmic_somatics.with_columns(
    CHROMOSOME=pl.col("CHROMOSOME").str.replace("MT","M")
    ).rename({"CHROMOSOME":"CHROMOSOME_1"}
             ).join(
                mutations_in_contigs_res.with_columns(
                    CHROMOSOME_1=pl.col("CHROMOSOME_1").str.strip_chars_start("chr")), 
                on=["GENOME_START","GENOME_STOP","CHROMOSOME_1"], 
                how="semi").collect()
setB.write_parquet(cosmic_mutations_in_contigs_semi_anti_file_B)


In [ ]:
# This is complimentary to setB
# ye just read the parquet as usual
%%script false --no-raise-error
non_contig_mutations = cosmic_somatics.with_columns(
    CHROMOSOME=pl.col("CHROMOSOME").str.replace("MT","M")
    ).rename({"CHROMOSOME":"CHROMOSOME_1"}
             ).join(
                mutations_in_contigs_res.with_columns(
                    CHROMOSOME_1=pl.col("CHROMOSOME_1").str.strip_chars_start("chr")), 
                on=["GENOME_START","GENOME_STOP","CHROMOSOME_1"], 
                how="anti").collect()

non_contig_mutations.write_parquet(cosmic_mutations_in_contigs_semi_anti_file_B_compliment)

In [ ]:
mutations_in_tq_res = pl.read_parquet(cosmic_somatics_overlap_with_top_quadrant_file)

In [ ]:
setB = pl.read_parquet(cosmic_mutations_in_contigs_semi_anti_file_B)


In [ ]:
non_contig_mutations = pl.read_parquet(cosmic_mutations_in_contigs_semi_anti_file_B_compliment)

In [12]:
setBtq = cosmic_somatics.with_columns(
    CHROMOSOME=pl.col("CHROMOSOME").str.replace("MT","M")
    ).rename({"CHROMOSOME":"CHROMOSOME_1"}
             ).join(
                mutations_in_tq_res.with_columns(
                    CHROMOSOME_1=pl.col("CHROMOSOME_1").str.strip_chars_start("chr")), 
                on=["GENOME_START","GENOME_STOP","CHROMOSOME_1"], 
                how="semi")
setBtq.write_parquet(cosmic_mutations_in_top_quadrant_semi_anti_file_B)
setBtq = pl.read_parquet(cosmic_mutations_in_top_quadrant_semi_anti_file_B)

non_tq_mutations = cosmic_somatics.with_columns(
    CHROMOSOME=pl.col("CHROMOSOME").str.replace("MT","M")
    ).rename({"CHROMOSOME":"CHROMOSOME_1"}
             ).join(
                mutations_in_tq_res.with_columns(
                    CHROMOSOME_1=pl.col("CHROMOSOME_1").str.strip_chars_start("chr")), 
                on=["GENOME_START","GENOME_STOP","CHROMOSOME_1"], 
                how="anti")

non_tq_mutations.write_parquet(cosmic_mutations_in_top_quadrant_semi_anti_file_B_compliment)
non_tq_mutations = pl.read_parquet(cosmic_mutations_in_top_quadrant_semi_anti_file_B_compliment)

# Calculating unique peptide promiscuities

In [ ]:
znow_bijatyka = pl.read_parquet(db_main_file)
unique_peptide_in_carmen_list = znow_bijatyka.unique("Peptide").select("Peptide")
for i in range(0,len(unique_peptide_in_carmen_list),10000):
    unique_peptide_in_carmen_list.slice(i,10000).write_parquet(temp_dir.joinpath(f"trashpanda{i}.parquet"))

In [ ]:
number_of_slices = math.ceil(len(unique_peptide_in_carmen_list)/10000)
# seq 0 10000 810000 | parallel --bar python ./calculate_promiscuity_peptides.py
for i in range(number_of_shards):
    os.system(f"python ./calculate_promiscuity_peptides.py {i}")

In [ ]:
trashpanda = pl.read_parquet(temp_dir.joinpath("bijatyka*"))
znow_bijatyka.join(trashpanda.rename({"score":"Promiscuity"}), on="Peptide").write_parquet(db_main_file, compression="zstd", compression_level=22)

# Final Cleanup

This is to clean up and delete all additional files and directories created throughout the analysis.

**Do not run the second cell unless you want to end your work here or start over.**

In [ ]:
# This is a safety code

raise KeyboardInterrupt("Are you sure you want to run the cell below?")

In [ ]:
cosmic_mutcount_per_contig_tmp.unlink()
unique_peptides_per_contig_tmp.unlink()
calculation_spam_dir.joinpath("*").unlink()
calculation_spam_dir.rmdir()
temp_dir.joinpath("*").unlink()
temp_dir.rmdir()
popcov_per_contig_file_tmp.unlink()
contig_hotspot_score_file_tmp.unlink()